# Brock s05_s06 (20260703) — automatic bout scoring from button-press events

The 20260703 session (s05/s06) has no hand-scored `ExpTrialBounds.mat`. Instead the
experimenter pressed the **event** sensor's button at the start and stop of every
movement bout, so the bounds live in the `.h5` `Annotations` stream ('Start'/'Stop').

Pipeline:

- **A. Events → snips** — `automatically_score_movements_from_events()` pairs Starts
  with Stops that occur within `BOUNDING_WINDOW_S`, reporting restarts (double-pressed
  Starts), orphan Stops, and suspiciously short pairs.
- **B. Condition table** — `load_condition_table()` parses the trialtable CSV and
  transcribes package size (ring=1, small=2, medium=3, large=4).
- **C. Align snips to the trial table** — `align_snips_to_trial_table()` measures each snip's walked distance
  from the foot IMUs (single-foot mechanization, max horizontal excursion, best foot
  wins and names the walker), expands the table into the expected bout sequence
  (each trial × 2 reps × 2 walkers), and sequence-aligns measured vs expected distance
  so **missed / skipped trials fall out as unmatched expected bouts**.

The cell in section A prints which foot IMUs the file actually contains — if one is
absent, that subject's bouts are measured from the remaining foot alone.


In [ ]:
# --- Google Colab setup (safe to run anywhere: it is a NO-OP locally) -------
# NO Google permissions are requested: the data comes in through link-shared
# file downloads, and results go out as a browser download (last cell).
#
# LAB SETUP (once, by the data owner): in Google Drive, right-click the
# session .h5 and the trialtable .csv -> Share -> 'Anyone with the link'
# (Viewer) -> Copy link, and paste the two links below. Students need nothing
# but this notebook's URL after that.
import os, sys
IN_COLAB = 'google.colab' in sys.modules

H5_URL = 'PASTE_DRIVE_LINK_TO_imuData_.h5_HERE'
TRIALTABLE_URL = 'PASTE_DRIVE_LINK_TO_trialtable_.csv_HERE'

if IN_COLAB:
    if not os.path.isdir('/content/stride_estimation_imu'):
        !git clone -q https://github.com/jeremydwong/stride_estimation_imu.git /content/stride_estimation_imu
    %cd /content/stride_estimation_imu/notebooks
    !pip install -q gdown ipympl
    from google.colab import output
    output.enable_custom_widget_manager()   # lets the interactive cells work
    import gdown
    os.makedirs('/content/brock_data', exist_ok=True)
    for url, name in ((H5_URL, None), (TRIALTABLE_URL, None)):
        if url.startswith('http'):
            gdown.download(url=url, output='/content/brock_data/', fuzzy=True)
        else:
            print('!! paste the Drive share links above first (ask the lab)')


In [ ]:
# --- setup / configuration ---
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']   # vector figures in the notebook
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../src'))   # stride_imu + brock_functions
import stride_imu as imu
from brock_functions import (load_events, automatically_score_movements_from_events,
                             load_condition_table, load_available_feet,
                             align_snips_to_trial_table, explain_missed, bout_spacing,
                             save_trial_figures, manual_correct,
                             apply_manual_rescore)

SESSION_TAG = 's05_s06'
# os.environ.get(NAME, default) = use the environment variable when one
# is set, else the default path. It never SETS the variable - it is a
# hook so a batch run can point this notebook at other files unedited.
# where the session data lives: the gdown folder on Colab, Dropbox locally
DATA_DIR = ('/content/brock_data' if 'google.colab' in sys.modules else
            '/Users/jeremy/Dropbox/Treadmill Brock 2025/imu data')
H5_FILE = os.environ.get('STRIDE_DATA_FILE',
    os.path.join(DATA_DIR, 'imuData_s05_s06_20260703.h5'))
CONDITION_CSV = os.environ.get('STRIDE_TRIALTABLE_FILE',
    os.path.join(DATA_DIR, 'trialtable_20260703.csv'))

BOUNDING_WINDOW_S = 30.0   # a Start and its Stop must fall within this window
MIN_BOUT_S = 1.5           # shorter pairs are flagged as likely accidental presses

# Recover bouts the experimenter STARTED but forgot to end: close them where
# the walker plants both feet (the hand-off stance). INFER_STOPS = False for
# clicks-only.
INFER_STOPS = True
QUIET_SECONDS = 0.75       # walker-pair stance that ends the bout: must be
                           # shorter than a hand-off stance (~0.9 s observed)
                           # and longer than within-gait double-stance (~0.3 s)
INFER_MIN_BOUT_S = 2.0     # ignore a settle this soon after the Start

# Alignment: how hard to penalise a match that puts the two bouts of ONE
# trial minutes apart. 0 = distance-only (the old behaviour).
TIME_WEIGHT = 0.6
pd.set_option('display.width', 160)


## A. Events → snips

Pair the button presses. The report shows every anomaly in the press stream:
`restarts` are Starts superseded by a later Start before any Stop (the bout keeps the
**last** Start), `orphan_stops` had no viable Start, and `short` flags sub-`MIN_BOUT_S`
pairs.

**Forgotten Stop clicks (`INFER_STOPS`).** Several bouts were started and never ended —
the experimenter got distracted, and the next press is another `Start`. Without
inference the earlier Start is discarded, silently losing a real bout. With
`INFER_STOPS = True`, `infer_stop_from_quiet()` closes it at the end of the **first
sustained walk**: per person (their own two feet only), still gaps shorter than
`QUIET_SECONDS` are absorbed into the motion run, so the walk ends exactly where the
walker plants both feet for `QUIET_SECONDS` — the hand-off stance. (Requiring all
FOUR feet still at once was the first attempt, and is wrong for this protocol: that
moment structurally never happens before the walker's un-clicked return walk, so
inferred bouts ran 30-40 s for 10 s walks.) The search runs only up to the next button
press so an inferred bout can never swallow the following one. A Start with no settle before that limit is still left unpaired rather than guessed.


In [ ]:
feet = load_available_feet(H5_FILE)
print('feet available:', list(feet), f'@ {1/next(iter(feet.values())).period:.0f} Hz')

events = load_events(H5_FILE)
snips, report = automatically_score_movements_from_events(
    events, bounding_window_s=BOUNDING_WINDOW_S, min_bout_s=MIN_BOUT_S,
    data=feet if INFER_STOPS else None, infer_stops=INFER_STOPS,
    quiet_seconds=QUIET_SECONDS, infer_min_bout_s=INFER_MIN_BOUT_S)
inferred = report['inferred']

durations = snips[:, 1] - snips[:, 0]
print(f"{report['n_start']} Starts + {report['n_stop']} Stops -> {report['n_snips']} snips "
      f"({report['n_inferred']} with an inferred stop)")
print(f"unrecovered duplicate Starts: {len(report['restarts'])} at t = {np.round(report['restarts'], 1)}")
print(f"orphan Stops: {np.round(report['orphan_stops'], 1)}")
print(f"short (<{MIN_BOUT_S}s) snips: {report['short']} -> {np.round(snips[report['short']], 1).tolist()}")
if report['n_inferred']:
    print(f"inferred-stop durations: median {np.median(durations[inferred]):.1f} s "
          f"vs {np.median(durations[~inferred]):.1f} s for clicked bouts")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 3))
ax0.plot(snips[~inferred, 0] / 60, durations[~inferred], '.', label='clicked')
ax0.plot(snips[inferred, 0] / 60, durations[inferred], 'D', mfc='none',
         color='#bf8700', ms=5, label='stop inferred')
for t in report['restarts']:
    ax0.axvline(t / 60, color='orange', lw=0.5)
for t in report['orphan_stops']:
    ax0.axvline(t / 60, color='red', lw=0.5)
ax0.set_xlabel('session time [min]'); ax0.set_ylabel('snip duration [s]')
ax0.set_title('snips (orange=unrecovered restart, red=orphan stop)')
ax0.legend(fontsize=8)
ax1.hist([durations[~inferred], durations[inferred]], bins=30, stacked=True,
         color=['#1f6feb', '#bf8700'], label=['clicked', 'inferred'])
ax1.set_xlabel('snip duration [s]'); ax1.set_title('duration distribution')
ax1.legend(fontsize=8)
plt.tight_layout()


## B. Condition table

48 trials; `package_code` transcribes ring=1, small=2, medium=3, large=4. The rep
status columns carry the experimenter's notes (interruptions etc.) — useful when
interpreting alignment anomalies below.


In [ ]:
conditions = load_condition_table(CONDITION_CSV)
print(conditions[conditions.rep1_status.ne('') | conditions.rep2_status.ne('')]
      [['trial', 'distance_m', 'package', 'rep1_status', 'rep2_status']].to_string(index=False))
conditions.head(8)


## C. Measure snip distances and compare to the trial table

This runs single-foot mechanization on every foot × snip (a few minutes). The
expected sequence is **rep-major** (confirmed against the data): the whole 48-trial
table is walked once (rep 1, 2 bouts per trial — one per walker), then again (rep 2).
The alignment tolerates extra snips and missing bouts.


In [ ]:
# drop the flagged sub-MIN_BOUT_S pairs (accidental presses) before aligning
keep = np.setdiff1d(np.arange(len(snips)), report['short'])
res = align_snips_to_trial_table(feet, snips[keep], conditions, inferred=inferred[keep],
                        time_weight=TIME_WEIGHT)
aligned = res['aligned']
err = aligned['distance_error_m'].abs()
print(f"matched {aligned['snip'].notna().sum()}/{len(aligned)} expected bouts; "
      f"median |distance error| {err.median():.2f} m (95th pct {err.quantile(0.95):.2f} m)")
print(f"of the matched bouts, {int(aligned['inferred'].sum())} had an inferred stop")


## D. Results — where were trials missed or skipped?

`missed` = expected bouts (trial/rep/walker-slot) with no matching snip.
`extra` = snips that matched no expected bout (aborted movements, stray presses).


In [ ]:
print('--- MISSED expected bouts ---')
print(res['missed'][['trial', 'rep', 'bout', 'distance_m', 'package', 'status']]
      .to_string(index=False))
print()
print('--- EXTRA snips (matched no expected bout) ---')
print(res['extra'][['start_s', 'stop_s', 'duration_s', 'distance_m', 'walker']]
      .round(2).to_string(index=False))


In [ ]:
# measured vs expected distance over the aligned sequence
fig, ax = plt.subplots(figsize=(13, 3.5))
x = np.arange(len(aligned))
ax.step(x, aligned['distance_m'], where='mid', color='0.6', label='expected')
ax.plot(x, aligned['measured_m'], '.', label='measured')
miss = aligned['snip'].isna()
ax.plot(x[miss], aligned['distance_m'][miss], 'rx', ms=9, label='missed')
for t in aligned['trial'].unique():
    ax.axvline(aligned.index[aligned['trial'] == t][0] - 0.5, color='0.9', lw=0.5, zorder=0)
ax.set_xlabel('expected bout sequence (4 per trial)'); ax.set_ylabel('distance [m]')
ax.legend(); ax.set_title('measured vs expected distance per bout')
plt.tight_layout()


## E. The same picture across EXPERIMENT TIME, over the button presses

The sequence plot above hides *where the clicks were*. Here the x axis is real
experiment time (the top axis keeps the expected bout-sequence index / trial), and every
button press is drawn underneath: **Start = up tick, Stop = down tick** — direction as
well as colour, so identity never depends on colour alone. Grey = expected,
blue = measured, red X = missed. Only two annotation types exist in these files
(`Start`/`Stop`, all from the `event` sensor) — there are no other button kinds.

Missed bouts have no snip, so each is placed at a time **interpolated** between its
matched neighbours. That is what makes the X readable against the press stream: if
presses sit under an X, the bout happened and the pairing/alignment rejected it; if the
band is empty under an X, the button was simply never pressed there.

**Pairing check (the 'uneven dots' flag):** each trial-rep should give exactly two bouts,
one per walker, at the same distance. The two are joined by a connector — **grey solid
when the pair is complete, red dashed when it is not**.


In [ ]:
# reaction time per bout (Start press -> snug gait start; ~10 s to compute).
# NEGATIVE = the walker was already going: the press was not a go cue -
# marked with an orange open triangle under the bout in the figure below.
from brock_functions import estimate_reaction_s
if 'reaction_s' not in aligned:
    aligned['reaction_s'] = estimate_reaction_s(feet, aligned)
print(f"{(aligned['reaction_s'] < 0).sum()} bouts with negative reaction "
      f"(median reaction {aligned['reaction_s'].median():+.2f} s)")

fig, axes = imu.draw_event_timeline(aligned, events, report, rows=6)

counts = imu.pair_status(aligned)
uneven = {k: v for k, v in counts.items() if v != 2}
print(f'trial-reps not yielding a clean pair: {len(uneven)}')
for (trial, rep), n in sorted(uneven.items()):
    print(f'  trial {trial:>2} rep {rep}: {n} of 2 bouts matched')


## F. Why is each bout missing? (documenting the logic)

*(The worked examples below — trials 37-40, trial 2 rep 2, etc. — are from the
s03/s04 20260622 session where this method was developed; the logic applies
unchanged here.)*

`align_snips_to_expected()` is a **Needleman-Wunsch global alignment** on distance
alone: match cost = `|measured - expected| / max(...)`, while skipping either an expected
bout ('missed') or a measured snip ('extra') costs a flat `gap_penalty` (0.55). It
returns the single cheapest path through the cost matrix.

**Identifiability caveat — important.** Because distance is the only signal, within a run
of equal-distance trials the alignment recovers *how many* bouts are missing but **not
which ones**. Trials 37-40 rep 1 are all 12 m: the table below attributes the gap to
trial 37/38, but that is a traceback tie-break (ties drift gaps toward the start of a
run), not evidence. The defensible claim is *'3 of the 8 twelve-metre bouts across trials
37-40 rep 1 are missing'*. Run boundaries are firm only where the distance changes.

**Time consistency (`TIME_WEIGHT`).** Distance alone is not enough: with a pure
distance cost the aligner happily placed the two bouts of trial 2 rep 2 **322 s**
apart (and trial 48 rep 1 **403 s** apart), bridging a hole while orphaning the five
genuine walks inside it as 'extra'. The two walkers of a rep actually go ~13 s apart,
so a diagonal step now also pays when it implies an impossible within-trial gap.
The penalty applies **only within a trial-rep** - real between-trial transitions vary
far too much to police, and penalising them made the alignment drop good matches
instead of fixing bad ones. This recovered trials 3-4 of rep 2 around t = 87-88 min.

`explain_missed()` adds two signals the aligner never sees:

- **presses in the gap** — 0 means the button was never pressed there; an odd count means
  one press of a pair went missing.
- **room in the clock** — the gap length against the session's own between-trial pacing.
  A gap no longer than a normal transition has no room for an extra walk, so a red X
  there is more likely an alignment artifact than a genuinely missed bout.

**A test that did NOT work, recorded so it is not retried:** foot motion during the gap.
It looks decisive but is not — the participants walk *back* to the start between trials,
so control transition gaps contain a median **11.2 m** of walking versus **9.1 m** in the
suspect gaps. `walked_m` is reported for context only and is excluded from the verdict.


In [ ]:
spacing = bout_spacing(aligned)
print(f"session pacing: {spacing['within_trial_s']:.0f} s between the two walkers of a rep, "
      f"{spacing['between_trial_s']:.0f} s between trials")

why = explain_missed(aligned, events, data=feet, spacing=spacing)
print(why[['trial', 'rep', 'bout', 'distance_m', 'gap_s', 'n_presses', 'presses',
           'walked_m', 'verdict']].round(1).to_string(index=False))
print()
print(why['verdict'].value_counts().to_string())

# sanity check: the two bouts of a trial-rep go one walker after the other (~13 s).
# A matched pair stretched far beyond that means the alignment bridged a hole
# instead of using the walks inside it - the failure the time term exists to stop.
m = aligned.dropna(subset=['start_s'])
pair_gaps = pd.DataFrame(
    [(t, r, g['start_s'].iloc[1] - g['stop_s'].iloc[0])
     for (t, r), g in m.groupby(['trial', 'rep']) if len(g) == 2],
    columns=['trial', 'rep', 'gap_s'])
print()
print(f"within-trial bout gap: median {pair_gaps.gap_s.median():.0f} s; "
      f"{(pair_gaps.gap_s > 60).sum()} pair(s) over 60 s")
print(pair_gaps.nlargest(3, 'gap_s').round(0).to_string(index=False))


In [ ]:
# save the aligned table beside the imu data (CACHE_DIR)
aligned.to_csv(ALIGNED_CSV, index=False)
print(f'wrote {ALIGNED_CSV},', len(aligned), 'rows')
aligned[aligned['snip'].isna() | aligned['distance_error_m'].abs().gt(1.5)]


## G. Per-trial visualization figures

One SVG per trial with every matched bout stacked (up to 2 reps × 2 walkers):
overhead foot map on the left, raw |A| / foot speed / step speed on the right —
the same panel as the dataset-1 velocity batch. Written to the
`figures/s05_s06/` folder **next to the session .h5** (in Dropbox, the same
`imu data/figures` the dataset-1 batch uses; created as needed) as
`brock_s05_s06_trial<N>_viz.svg` — kept out of the git repo.
This runs the two-IMU stride pipeline per bout, so expect several minutes for
all 48 trials; set `FIG_TRIALS` to a short list (e.g. `[1, 2]`) while testing.


In [ ]:
FIG_TRIALS = None    # None = all trials; or a list like [1, 2, 37]
# figures live in a "figures" folder NEXT TO THE IMU DATA (not in the repo)
FIGURES_DIR = os.path.join(os.path.dirname(H5_FILE), 'figures')

fig_paths = save_trial_figures(feet, aligned, session_tag=SESSION_TAG,
                               trials=FIG_TRIALS, out_dir=FIGURES_DIR)
print(f'{len(fig_paths)} trial figures -> {os.path.dirname(fig_paths[0])}')


## H. Manual inspection & click-to-rescore

**Prefer the draggable inspector** (tutorial cell 6 / `interactive_inspect_trial`) for bouts
that exist but have bad bounds — drag the gait start / bout end and press *save
adjustments*. `manual_correct` below is mainly for **missed** bouts: with no snip
there is nothing to drag, and its window (placed from the interpolated bout time)
lets you click a start/stop where the alignment found none.

When a bout above looks wrong (a `missed` verdict you disagree with, a nonsense
duration, an `[inferred stop]` figure showing extra walking), list its trial here
and rescore it by clicking. For each trial-rep an inspection figure shows raw |A|,
the mechanized foot speed, and horizontal excursion for every foot, with the
current bout windows shaded (person A blue, person B orange — a missed bout has
no shading). On the figure:

1. press **rescore A** (or **rescore B**) — usually only one person's bout needs
   fixing, so each is rescored separately;
2. **click the plot twice**: first click = the new bout START, second = STOP;
3. press **save** — the correction is appended to
   `brock_s05_s06_manual_rescore.csv` (nothing is written until you press save).

The **from [s]** / **to [s]** text fields set the displayed time window —
type a number and press Enter to re-slice and redraw (useful when the
walking you care about sits outside the default window).

Afterwards `apply_manual_rescore(aligned, csv)` folds the saved corrections back
into the table (last save wins if you rescored twice).

### If the interactive figure does not appear / errors out

**Easiest reliable path — skip Jupyter entirely.** The same click-to-rescore
figures open as native windows from the terminal (from the repo folder):

```
uv run python rescore_brock.py s05_s06 1 37:2
```

(each argument is a trial — `1` shows both reps, `37:2` just rep 2; close each
window to move to the next; press **save** before closing if you rescored).
Corrections land in the same CSV, so the fold-in cell below works unchanged.
Run the notebook once first — the script reads the `brock_<tag>_auto_aligned.csv`
it writes.

If you'd rather stay in the notebook, `manual_correct` checks the environment
BEFORE drawing anything and its error message prints which python the kernel is
on and what to fix:

* **`ModuleNotFoundError: ipympl` / "Could not switch to the interactive
  'widget' backend"** → the notebook is running on the WRONG PYTHON. The
  kernel must be this project's `.venv`: in VS Code, kernel picker (top-right)
  → *Select Another Kernel* → *Python Environments* → **`.venv`**, then
  *Restart* the kernel (↻) and re-run from the top. If the kernel already IS
  `.venv`, run `uv sync` in a terminal, then restart the kernel.
* **VS Code still won't go interactive** (it often won't) → use the terminal
  script above, or the browser: `uv run jupyter lab` from the repo folder.
* **Clicks do nothing** → the zoom/pan tool in the figure toolbar is switched
  on; click its icon to turn it off, then click in the plot again.

**After editing anything in `brock_functions.py`** (or pulling changes):
restart the kernel — Python caches imported modules, so a re-run without a
restart keeps executing the old code.


In [ ]:
MANUAL_INSPECT = []   # trials to inspect, e.g. [1, (37, 2)]
                      # a bare number shows both reps; (trial, rep) just one

if MANUAL_INSPECT:
    controllers = manual_correct(   # keep the return value! (buttons die otherwise)
        feet, aligned, MANUAL_INSPECT,
        out_csv=RESCORE_CSV,
        session_tag=SESSION_TAG)
else:
    print('MANUAL_INSPECT is empty - nothing to inspect. Add trial numbers'
          ' (or (trial, rep) tuples) to the list above and re-run this cell.')


In [ ]:
# after rescoring + saving above, fold the corrections into the table:
csv = RESCORE_CSV
if os.path.exists(csv):
    aligned_fixed = apply_manual_rescore(aligned, csv)
    aligned_fixed.to_csv(ALIGNED_CSV, index=False)
    print(f"{int(aligned_fixed['manual'].sum())} manually-rescored bout(s) "
          f'folded in and saved to {ALIGNED_CSV}')
else:
    print('no manual corrections saved yet')


## I. Student playground — the three data types

Everything in this notebook is functions applied to **three kinds of data**.
If you are comfortable with these, you can re-analyse anything:

1. **Raw IMU recordings** — `feet` is a plain dict `{label -> ImuRecording}`
   (`'left_foot_a'`, `'right_foot_b'`, ...). An `ImuRecording` holds the gyro
   `Wb` (rad/sample), the accelerometer `Ab` (m/s²) and the sample `period`
   (s); `rec[a:b]` slices it like a list. Every trajectory, footfall and speed
   in this notebook is *derived* from these two signals.
2. **Tables** — `conditions` (one row per trial: distance, package, pose,
   experimenter notes) and `aligned` (one row per expected bout: when it
   happened, measured distance, who walked). Both are ordinary pandas
   DataFrames: filter, merge, group, plot.
3. **Rescore controllers** — the return value of `manual_correct()`: one
   object per inspected trial-rep that knows its window, the current bout
   spans and any rescores you clicked. `print()` one to see its state.

The cells below poke at each in turn — copy, edit and re-run them freely.

None of this needs memorizing — human working memory holds ~7 items, and these
objects hold far more. Cell (0) below shows the three introspection patterns
(`list(d)` / `.items()` for dicts, `vars(obj)` for objects, `.dtypes`/`.head()`
for tables) that let you ask *any* variable what it contains.


In [ ]:
# --- 0) don't memorize - ask the object what it contains --------------------
# Human working memory holds ~7 items; this notebook's objects hold far more.
# So never try to remember what is inside something - ASK it. Three patterns
# cover every named variable in this notebook:

# (a) a DICT, like `feet`: list its keys, or loop over key -> value pairs
print('feet is a', type(feet).__name__, 'with keys:', list(feet))
for name, rec in feet.items():
    print(f'   {name}: {len(rec)} samples @ {1/rec.period:.0f} Hz')

# (b) an OBJECT, like one ImuRecording: vars(obj) is a dict of its fields.
# Print names + types + shapes instead of the values (arrays are huge):
rec = feet['left_foot_a']
print('\none ImuRecording contains:')
for field, value in vars(rec).items():
    desc = getattr(value, 'shape', None) or type(value).__name__
    print(f'   rec.{field:22s} {desc}')
# (for anything else: dir(obj) lists methods too, help(obj) prints the docs)

# (c) a DataFrame, like `aligned` or `conditions`: .columns/.dtypes name the
# columns, .head() peeks at rows, .describe() summarizes numeric columns
print('\naligned columns:'); print(aligned.dtypes.to_string())
aligned.head(3)


In [ ]:
# --- 1) raw IMU data: pick a bout, look at the actual signals ---------------
rec = feet['left_foot_a']
print(f"left_foot_a: {len(rec)} samples @ {1/rec.period:.0f} Hz "
      f"({len(rec)*rec.period/60:.1f} min of recording)")

bout = aligned.dropna(subset=['start_s']).iloc[0]      # first scored bout
j0, j1 = int(bout.start_s / rec.period), int(bout.stop_s / rec.period)
piece = rec[j0:j1]                                     # slicing = new recording
t = np.arange(len(piece)) * rec.period

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t, np.linalg.norm(piece.Ab, axis=1), lw=0.6)
axes[0].set_ylabel('|A| [m/s²]')
axes[1].plot(t, np.linalg.norm(piece.Wb, axis=1) / rec.period, lw=0.6)
axes[1].set(ylabel='|gyro| [rad/s]', xlabel='time in bout [s]')
fig.suptitle(f"trial {bout.trial:.0f} rep {bout.rep:.0f} bout {bout.bout:.0f}: "
             f"raw left_foot_a signals")
plt.show()


In [ ]:
# --- 2) re-run the mechanization yourself -----------------------------------
# compute_position_two_imus integrates gyro + accel into foot trajectories,
# applying zero-velocity updates at every detected stance. It returns one
# FootTrajectory per foot: .P (position, m), .Vm (speed, m/s), .FF_walking
# (footfall mask), .euler ... — this is the engine under every figure above.
suffix = str(bout.walker)[-1]          # 'a' or 'b': who walked this bout
L = feet[f'left_foot_{suffix}'][j0:j1]
R = feet[f'right_foot_{suffix}'][j0:j1]
left_info, right_info = imu.compute_position_two_imus(L.Wb, L.Ab, R.Wb, R.Ab,
                                                      L.period)

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t, left_info.Vm, lw=0.7, label='left')
axes[0].plot(t, right_info.Vm, lw=0.7, label='right')
axes[0].set_ylabel('foot speed [m/s]'); axes[0].legend()
axes[1].plot(t, np.linalg.norm(left_info.P[:, :2] - left_info.P[0, :2], axis=1),
             lw=0.8, label='left')
axes[1].plot(t, np.linalg.norm(right_info.P[:, :2] - right_info.P[0, :2], axis=1),
             lw=0.8, label='right')
axes[1].set(ylabel='distance from start [m]', xlabel='time in bout [s]')
plt.show()
print(f"net horizontal travel: "
      f"left {np.linalg.norm(left_info.P[-1,:2]-left_info.P[0,:2]):.2f} m, "
      f"right {np.linalg.norm(right_info.P[-1,:2]-right_info.P[0,:2]):.2f} m "
      f"(trial table says {bout.distance_m} m)")


In [ ]:
# --- 3) tables: merge results with conditions, plot anything vs anything ----
# `aligned` already carries distance_m/package per bout; the merge adds the
# hand-off pose and the experimenter's per-rep notes from `conditions`.
table = (aligned.dropna(subset=['start_s'])
         .merge(conditions[['trial', 'pose', 'rep1_status', 'rep2_status']],
                on='trial'))
# crude bout-average speed: walked distance over the whole scored window
# (includes box handling — the per-STEP speeds in the figures are the real
# gait measure; this is just an easy table exercise)
table['bout_speed_mps'] = table['measured_m'] / table['duration_s']

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
table.boxplot(column='bout_speed_mps', by='distance_m', ax=axes[0])
table.boxplot(column='bout_speed_mps', by='pose', ax=axes[1])
for ax in axes:
    ax.set_ylabel('bout speed [m/s]'); ax.set_title('')
fig.suptitle('bout-average speed by condition')
plt.tight_layout(); plt.show()

table.groupby('distance_m')['bout_speed_mps'].agg(['mean', 'std', 'count']).round(2)


In [ ]:
# --- 5) the per-trial inspection figure, on demand --------------------------
# inspect_snipped_trial draws ONE trial's full figure (all matched bouts) and
# RETURNS the matplotlib Figure: grey pre-walk accel context, the Start/Stop
# button presses as vertical lines, and the reaction time / duration written
# on each bout. Change the trial number and re-run. Full details:
# help(inspect_snipped_trial)
from brock_functions import inspect_snipped_trial

fig = inspect_snipped_trial(feet, aligned, trial=5, session_tag=SESSION_TAG,
                            prefix_seconds=5.0)


In [ ]:
# --- 6) interactive: DRAG the gait start / bout end -------------------------
# Same figure as (5) but live: the GREEN line is the snug gait start, the
# PURPLE line the bout end. Mouse-down grabs whichever is closer, drag it,
# release -> that bout re-runs with the adjusted bounds and redraws (the
# time axis re-zeroes to the new gait start). The ORIGINAL clicks are never
# modified: press 'save adjustments' (bottom right) to append your edits to
# the manual-rescore CSV - the fold-in cell below then stamps them
# manual=True with the person. KEEP the return value or dragging dies.
# Needs the interactive backend - same setup notes as section H.
from brock_functions import interactive_inspect_trial

drag_ctrl = interactive_inspect_trial(
    feet, aligned, trial=5, session_tag=SESSION_TAG,
    out_csv=RESCORE_CSV)


In [ ]:
# --- 4) rescore controllers (from section H's manual_correct) ---------------
# With the draggable inspector above, you mostly do NOT need manual_correct
# for bouts that exist but have bad bounds - just drag them. manual_correct's
# remaining job is MISSED bouts: when no snip exists there is nothing to drag,
# and its inspection window (placed from the interpolated bout time) lets you
# click a start/stop where the alignment found none. Its controllers know
# their state - print one to see it:
if 'controllers' in dir() and controllers:
    for c in controllers:
        print(c)          # trial/rep, shown window, bout spans, your rescores
else:
    print('no controllers - only needed for MISSED bouts; run the '
          'MANUAL_INSPECT cell in section H with a trial that has a red X')


In [ ]:
# --- Colab only: download everything this notebook produced -----------------
# Zips the cached tables + this session's figures (they live in the Colab VM,
# which is wiped when the session ends) and hands the zip to your browser.
# No Google permissions involved. A no-op when running locally.
if IN_COLAB:
    import zipfile
    from google.colab import files
    zpath = f'/content/brock_{SESSION_TAG}_results.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for sub in ('cached data', 'figures'):     # outputs only, not the .h5
            for root, _, fns in os.walk(os.path.join(DATA_DIR, sub)):
                for fn in fns:
                    p = os.path.join(root, fn)
                    z.write(p, os.path.relpath(p, DATA_DIR))
    files.download(zpath)
else:
    print('running locally - results are already in', os.path.dirname(H5_FILE))
